In [ ]:
# =============================================================
# 05_task_A_prediction.ipynb
# Task A — Outcome Prediction Baselines
# ------------------------------------------------------------
# Benchmark task: predict lower-secondary completion rate Y_it
# from lagged covariates X_{i,t-1} under two evaluation
# protocols (temporal split and LOCO).
#
# Models
# ──────
#   1. Pooled OLS with fixed-effects dummies (FE-OLS)
#   2. Ridge regression (L2-penalised OLS)
#   3. Random Forest regressor
#   4. XGBoost regressor
#   5. Naïve baseline (country mean / last-observation)
#
# Feature engineering
# ───────────────────
#   - All 16 non-outcome indicators as covariates (lagged 1 year)
#   - Country and year one-hot dummies for FE-OLS
#   - Standard scaling (mean=0, std=1) for Ridge; tree models unscaled
#   - Missing covariate values imputed with column median (train set)
#
# Metrics
# ───────
#   RMSE, MAE, R² — reported per model × split protocol
#
# Outputs
# ───────
#   outputs/results/task_A_metrics.csv
#   outputs/results/task_A_predictions.csv   (all folds, all models)
#   outputs/tables/table_taskA.csv / .tex
#   outputs/figures/panel_12/taskA_actual_vs_pred.png
#   outputs/figures/panel_12/taskA_rmse_comparison.png
#
# Usage
# ─────
#   python src/05_task_A_prediction.py
#
# Requirements
# ────────────
#   pip install scikit-learn pandas numpy matplotlib pyyaml
#   pip install xgboost   (optional — skipped gracefully if absent)
# =============================================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

from utils import find_project_root, load_indicators, load_pipeline

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR  = DATA_DIR / "train_test_splits"
RESULT_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR  = PROJECT_ROOT / "outputs" / "tables"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

for d in [RESULT_DIR, TABLE_DIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

# Outcome and feature columns
OUTCOME   = "sec_completion"
FEATURES  = [k for k, v in IND_CFG["indicators"].items()
             if k != OUTCOME and v.get("role") != "C"]
# Remove pop_total (role=C) — not a causal covariate

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Outcome       : {OUTCOME}")
print(f"[info] Features      : {len(FEATURES)} indicators")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    panel_path  = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    temporal_p  = SPLIT_DIR / "temporal_split.csv"
    loco_p      = SPLIT_DIR / "loco_splits.csv"

    for p in [panel_path, temporal_p, loco_p]:
        if not p.exists():
            raise FileNotFoundError(f"Required file not found: {p}")

    panel    = pd.read_csv(panel_path)
    temporal = pd.read_csv(temporal_p)
    loco     = pd.read_csv(loco_p)

    print(f"\n[step 0] Loaded panel  : {len(panel)} rows")
    print(f"         temporal split: {(temporal['split']=='train').sum()} train / "
          f"{(temporal['split']=='test').sum()} test")
    print(f"         LOCO folds    : {loco['fold'].nunique()}")
    return panel, temporal, loco


# ──────────────────────────────────────────────────────────────
# Feature engineering — lagged covariates
# ──────────────────────────────────────────────────────────────
def make_lagged_features(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Create one-year lagged versions of all feature columns.
    Y_it is the current-year outcome.
    X_{i,t-1} are the lagged covariates.

    Returns panel with added lag columns and drops year=YEAR_MIN
    (no lag available for the first year).
    """
    lag_cols = {}
    for feat in FEATURES:
        if feat in panel.columns:
            lag_cols[f"{feat}_lag1"] = (
                panel.groupby("iso3c")[feat].shift(1)
            )

    panel_lag = panel.copy()
    for name, series in lag_cols.items():
        panel_lag[name] = series

    # Drop first year — no lag available
    panel_lag = panel_lag[panel_lag["year"] > YEAR_MIN].copy()
    print(f"\n[step 1] Lagged features: {len(lag_cols)} lag columns")
    print(f"         Rows after dropping year {YEAR_MIN}: {len(panel_lag)}")
    return panel_lag


# ──────────────────────────────────────────────────────────────
# Prepare X, y arrays
# ──────────────────────────────────────────────────────────────
def get_feature_cols(panel_lag: pd.DataFrame) -> list[str]:
    return [c for c in panel_lag.columns if c.endswith("_lag1")]


def prepare_Xy(
    panel_lag: pd.DataFrame,
    row_mask:  pd.Series,
    feature_cols: list[str],
    imputer:   SimpleImputer | None = None,
    fit_imputer: bool = False,
) -> tuple[np.ndarray, np.ndarray, SimpleImputer]:
    """
    Extract X (features) and y (outcome) for rows matching row_mask.
    Missing features imputed with column median (fitted on train, applied to test).
    Rows where outcome is NaN are dropped.
    """
    sub = panel_lag[row_mask].copy()
    sub = sub.dropna(subset=[OUTCOME])

    X_raw = sub[feature_cols].values.astype(float)
    y     = sub[OUTCOME].values.astype(float)

    if imputer is None or fit_imputer:
        imputer = SimpleImputer(strategy="median")
        X       = imputer.fit_transform(X_raw)
    else:
        X       = imputer.transform(X_raw)

    return X, y, imputer


# ──────────────────────────────────────────────────────────────
# Model registry
# ──────────────────────────────────────────────────────────────
def get_models() -> dict:
    models: dict[str, object] = {
        "Ridge":         Ridge(alpha=1.0),
        "RandomForest":  RandomForestRegressor(
                             n_estimators=200,
                             max_depth=5,
                             min_samples_leaf=3,
                             random_state=42,
                             n_jobs=-1,
                         ),
        "Naive_Mean":    None,   # handled separately
        "Naive_LastObs": None,   # handled separately
    }
    # XGBoost — optional
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0,
        )
        print("  XGBoost available — included")
    except ImportError:
        print("  XGBoost not installed — skipped  (pip install xgboost to enable)")
    return models


# ──────────────────────────────────────────────────────────────
# Naive baselines
# ──────────────────────────────────────────────────────────────
def naive_mean_pred(
    panel_lag: pd.DataFrame,
    train_mask: pd.Series,
    test_mask:  pd.Series,
) -> np.ndarray:
    """Predict country mean of outcome computed on train rows."""
    train_means = (
        panel_lag[train_mask]
        .dropna(subset=[OUTCOME])
        .groupby("iso3c")[OUTCOME]
        .mean()
    )
    test_sub  = panel_lag[test_mask].dropna(subset=[OUTCOME])
    preds = test_sub["iso3c"].map(train_means)
    # For countries not in train (possible in LOCO), use global train mean
    global_mean = panel_lag[train_mask][OUTCOME].mean()
    preds = preds.fillna(global_mean)
    return preds.values


def naive_lastobs_pred(
    panel_lag:  pd.DataFrame,
    train_mask: pd.Series,
    test_mask:  pd.Series,
) -> np.ndarray:
    """Predict last observed value from train for each country."""
    last_obs = (
        panel_lag[train_mask]
        .dropna(subset=[OUTCOME])
        .sort_values("year")
        .groupby("iso3c")[OUTCOME]
        .last()
    )
    test_sub = panel_lag[test_mask].dropna(subset=[OUTCOME])
    preds = test_sub["iso3c"].map(last_obs)
    global_last = panel_lag[train_mask][OUTCOME].dropna().iloc[-1]
    preds = preds.fillna(global_last)
    return preds.values


# ──────────────────────────────────────────────────────────────
# Metrics helper
# ──────────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    return {"rmse": round(rmse, 4), "mae": round(mae, 4), "r2": round(r2, 4)}


# ──────────────────────────────────────────────────────────────
# Temporal split evaluation
# ──────────────────────────────────────────────────────────────
def run_temporal(
    panel_lag:    pd.DataFrame,
    temporal:     pd.DataFrame,
    feature_cols: list[str],
    models:       dict,
) -> list[dict]:
    print("\n[step 2] Temporal split evaluation …")

    train_isos = set(temporal.loc[temporal["split"]=="train","iso3c"])
    test_isos  = set(temporal.loc[temporal["split"]=="test", "iso3c"])
    train_yrs  = set(temporal.loc[temporal["split"]=="train","year"])
    test_yrs   = set(temporal.loc[temporal["split"]=="test", "year"])

    train_mask = (panel_lag["iso3c"].isin(train_isos) &
                  panel_lag["year"].isin(train_yrs))
    test_mask  = (panel_lag["iso3c"].isin(test_isos) &
                  panel_lag["year"].isin(test_yrs))

    X_tr, y_tr, imputer = prepare_Xy(
        panel_lag, train_mask, feature_cols, fit_imputer=True)
    X_te, y_te, _       = prepare_Xy(
        panel_lag, test_mask, feature_cols, imputer=imputer)

    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_te_sc  = scaler.transform(X_te)

    records = []

    for name, model in models.items():
        if model is None:
            # Naive baselines
            if name == "Naive_Mean":
                y_pred = naive_mean_pred(panel_lag, train_mask, test_mask)
            else:
                y_pred = naive_lastobs_pred(panel_lag, train_mask, test_mask)
            m = compute_metrics(y_te, y_pred)
        else:
            # Scale only for Ridge
            Xtr = X_tr_sc if name == "Ridge" else X_tr
            Xte = X_te_sc if name == "Ridge" else X_te
            model.fit(Xtr, y_tr)
            y_pred = model.predict(Xte)
            m = compute_metrics(y_te, y_pred)

        print(f"  {name:<18}  RMSE={m['rmse']:.3f}  MAE={m['mae']:.3f}  "
              f"R²={m['r2']:.3f}")
        records.append({
            "protocol": "temporal",
            "fold":     "—",
            "model":    name,
            **m,
            "n_train":  int(y_tr.shape[0]),
            "n_test":   int(y_te.shape[0]),
        })

    return records


# ──────────────────────────────────────────────────────────────
# LOCO evaluation
# ──────────────────────────────────────────────────────────────
def run_loco(
    panel_lag:    pd.DataFrame,
    loco:         pd.DataFrame,
    feature_cols: list[str],
    models:       dict,
) -> list[dict]:
    print("\n[step 3] LOCO evaluation …")

    all_records = []
    folds = sorted(loco["fold"].unique())

    for fold in folds:
        fold_data  = loco[loco["fold"] == fold]
        held_iso   = fold_data["held_out_iso3c"].iloc[0]
        held_name  = fold_data["held_out_country"].iloc[0]

        train_keys = set(zip(
            fold_data.loc[fold_data["split"]=="train","iso3c"],
            fold_data.loc[fold_data["split"]=="train","year"],
        ))
        test_keys  = set(zip(
            fold_data.loc[fold_data["split"]=="test","iso3c"],
            fold_data.loc[fold_data["split"]=="test","year"],
        ))

        train_mask = panel_lag.apply(
            lambda r: (r["iso3c"], r["year"]) in train_keys, axis=1)
        test_mask  = panel_lag.apply(
            lambda r: (r["iso3c"], r["year"]) in test_keys,  axis=1)

        X_tr, y_tr, imputer = prepare_Xy(
            panel_lag, train_mask, feature_cols, fit_imputer=True)
        X_te, y_te, _       = prepare_Xy(
            panel_lag, test_mask, feature_cols, imputer=imputer)

        if len(y_te) == 0:
            continue

        scaler   = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_tr)
        X_te_sc  = scaler.transform(X_te)

        for name, model in models.items():
            if model is None:
                if name == "Naive_Mean":
                    y_pred = naive_mean_pred(panel_lag, train_mask, test_mask)
                else:
                    y_pred = naive_lastobs_pred(panel_lag, train_mask, test_mask)
                m = compute_metrics(y_te, y_pred)
            else:
                # Clone model to avoid state leakage between folds
                from sklearn.base import clone as sk_clone
                m_clone = sk_clone(model)
                Xtr = X_tr_sc if name == "Ridge" else X_tr
                Xte = X_te_sc if name == "Ridge" else X_te
                m_clone.fit(Xtr, y_tr)
                y_pred = m_clone.predict(Xte)
                m = compute_metrics(y_te, y_pred)

            all_records.append({
                "protocol":    "loco",
                "fold":        fold,
                "held_out":    held_iso,
                "held_name":   held_name,
                "model":       name,
                **m,
                "n_train":     int(y_tr.shape[0]),
                "n_test":      int(y_te.shape[0]),
            })

    # Aggregate LOCO means
    agg = (
        pd.DataFrame(all_records)
        .groupby("model")[["rmse","mae","r2","n_train","n_test"]]
        .mean()
        .round(4)
        .reset_index()
    )
    print(f"\n  LOCO mean metrics (averaged over {len(folds)} folds):")
    for _, row in agg.iterrows():
        print(f"  {row['model']:<18}  RMSE={row['rmse']:.3f}  "
              f"MAE={row['mae']:.3f}  R²={row['r2']:.3f}")

    return all_records


# ──────────────────────────────────────────────────────────────
# Feature importance (Random Forest)
# ──────────────────────────────────────────────────────────────
def get_feature_importance(
    panel_lag:    pd.DataFrame,
    temporal:     pd.DataFrame,
    feature_cols: list[str],
) -> pd.DataFrame:
    """
    Fit RF on full training set (temporal split) and extract
    mean decrease impurity importance.
    """
    train_isos = set(temporal.loc[temporal["split"]=="train","iso3c"])
    train_yrs  = set(temporal.loc[temporal["split"]=="train","year"])
    train_mask = (panel_lag["iso3c"].isin(train_isos) &
                  panel_lag["year"].isin(train_yrs))

    X_tr, y_tr, _ = prepare_Xy(
        panel_lag, train_mask, feature_cols, fit_imputer=True)

    rf = RandomForestRegressor(
        n_estimators=500, max_depth=5,
        min_samples_leaf=3, random_state=42, n_jobs=-1,
    )
    rf.fit(X_tr, y_tr)

    imp = pd.DataFrame({
        "feature":    feature_cols,
        "importance": rf.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    # Strip _lag1 suffix and map to display name
    from utils import load_indicators as _li
    _ind = _li()["indicators"]
    imp["indicator"] = imp["feature"].str.replace("_lag1","",regex=False)
    imp["label"]     = imp["indicator"].map(
        lambda k: _ind.get(k,{}).get("label", k)
    )
    return imp


# ──────────────────────────────────────────────────────────────
# Figures
# ──────────────────────────────────────────────────────────────
def plot_rmse_comparison(all_records: list[dict]) -> None:
    """Bar chart: RMSE by model × protocol."""
    df = pd.DataFrame(all_records)

    # Temporal results
    temp_df = df[df["protocol"]=="temporal"][["model","rmse","mae","r2"]]

    # LOCO mean results
    loco_df = (
        df[df["protocol"]=="loco"]
        .groupby("model")[["rmse","mae","r2"]]
        .mean()
        .reset_index()
    )

    models_order = temp_df["model"].tolist()
    # Ensure consistent order
    loco_df = loco_df.set_index("model").reindex(models_order).reset_index()
    temp_df = temp_df.set_index("model").reindex(models_order).reset_index()

    x      = np.arange(len(models_order))
    width  = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                              gridspec_kw={"wspace": 0.35})

    for ax, metric, ylabel, title_sfx in [
        (axes[0], "rmse", "RMSE (pp)", "RMSE"),
        (axes[1], "r2",   "R²",        "R²"),
    ]:
        bars1 = ax.bar(x - width/2, temp_df[metric], width,
                       label="Temporal", color="#4393c3", alpha=0.85,
                       edgecolor="white")
        bars2 = ax.bar(x + width/2, loco_df[metric], width,
                       label="LOCO mean", color="#d6604d", alpha=0.85,
                       edgecolor="white")

        ax.set_xticks(x)
        ax.set_xticklabels(models_order, rotation=30, ha="right", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f"Task A — {title_sfx} by model and protocol",
                     fontsize=10, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.25, lw=0.6)
        ax.spines[["top","right"]].set_visible(False)

        # Annotate bar values
        for bar in [*bars1, *bars2]:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                        f"{h:.2f}", ha="center", va="bottom", fontsize=7)

    fig.suptitle("Task A — Outcome prediction baselines\n"
                 "Lower-secondary completion rate (2015–2024)",
                 fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    out = FIGDIR / f"taskA_rmse_comparison_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_actual_vs_pred(
    panel_lag:    pd.DataFrame,
    temporal:     pd.DataFrame,
    feature_cols: list[str],
) -> None:
    """
    Scatter: actual vs predicted (temporal test set),
    best model (lowest RMSE) vs naive mean.
    """
    train_isos = set(temporal.loc[temporal["split"]=="train","iso3c"])
    test_isos  = set(temporal.loc[temporal["split"]=="test", "iso3c"])
    train_yrs  = set(temporal.loc[temporal["split"]=="train","year"])
    test_yrs   = set(temporal.loc[temporal["split"]=="test", "year"])

    train_mask = (panel_lag["iso3c"].isin(train_isos) &
                  panel_lag["year"].isin(train_yrs))
    test_mask  = (panel_lag["iso3c"].isin(test_isos) &
                  panel_lag["year"].isin(test_yrs))

    X_tr, y_tr, imputer = prepare_Xy(
        panel_lag, train_mask, feature_cols, fit_imputer=True)
    X_te, y_te, _       = prepare_Xy(
        panel_lag, test_mask, feature_cols, imputer=imputer)

    test_sub = panel_lag[test_mask].dropna(subset=[OUTCOME])

    # Random Forest predictions
    rf = RandomForestRegressor(n_estimators=200, max_depth=5,
                                min_samples_leaf=3, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    y_rf = rf.predict(X_te)

    # Naive mean
    y_naive = naive_mean_pred(panel_lag, train_mask, test_mask)

    # Plot
    TIER_COLOURS = {
        "High income":         "#1a6fad",
        "Upper middle income": "#1a7a3a",
        "Lower middle income": "#d73027",
        "Low income":          "#7b2d8b",
    }

    fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                              gridspec_kw={"wspace": 0.35})

    for ax, y_pred, title in [
        (axes[0], y_rf,    "Random Forest"),
        (axes[1], y_naive, "Naïve Mean Baseline"),
    ]:
        for _, row, yp in zip(
            range(len(test_sub)),
            test_sub.itertuples(),
            y_pred,
        ):
            col = TIER_COLOURS.get(row.income_group, "grey")
            ax.scatter(row.sec_completion, yp, color=col,
                       s=55, alpha=0.75, edgecolors="white",
                       linewidths=0.5, zorder=3)

        # 45° line
        lim = [0, 110]
        ax.plot(lim, lim, color="#888888", lw=1.2, ls="--", zorder=1)

        m = compute_metrics(y_te, y_pred)
        ax.set_title(
            f"{title}\nRMSE={m['rmse']:.2f} pp  R²={m['r2']:.3f}",
            fontsize=10, fontweight="bold",
        )
        ax.set_xlabel("Actual completion rate (%)", fontsize=10)
        ax.set_ylabel("Predicted completion rate (%)", fontsize=10)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[["top","right"]].set_visible(False)

    # Tier legend
    from matplotlib.patches import Patch
    handles = [Patch(color=c, label=t) for t, c in TIER_COLOURS.items()]
    fig.legend(handles=handles, loc="lower center", ncol=4,
               fontsize=8, bbox_to_anchor=(0.5, -0.06),
               frameon=True, edgecolor="#cccccc")

    fig.suptitle("Task A — Actual vs Predicted (temporal test set 2022–2024)",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    out = FIGDIR / f"taskA_actual_vs_pred_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_feature_importance(imp: pd.DataFrame) -> None:
    """Horizontal bar chart of top-15 RF feature importances."""
    top = imp.head(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top["label"][::-1], top["importance"][::-1],
            color="#4393c3", edgecolor="white", height=0.65)
    ax.set_xlabel("Mean decrease in impurity (normalised)", fontsize=10)
    ax.set_title("Task A — Random Forest feature importance (top 15, lag-1 features)",
                 fontsize=10, fontweight="bold")
    ax.grid(axis="x", alpha=0.25, lw=0.6)
    ax.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskA_feature_importance_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save results
# ──────────────────────────────────────────────────────────────
def save_results(all_records: list[dict]) -> None:
    df = pd.DataFrame(all_records)

    # Full predictions log
    df.to_csv(RESULT_DIR / "task_A_metrics.csv", index=False)
    print(f"  [OK] task_A_metrics.csv  ({len(df)} rows)")

    # Summary table: temporal + LOCO mean
    temp = df[df["protocol"]=="temporal"][
        ["model","rmse","mae","r2","n_train","n_test"]
    ].rename(columns={"rmse":"rmse_temp","mae":"mae_temp",
                       "r2":"r2_temp"})

    loco_mean = (
        df[df["protocol"]=="loco"]
        .groupby("model")[["rmse","mae","r2"]]
        .mean().round(4).reset_index()
        .rename(columns={"rmse":"rmse_loco_mean","mae":"mae_loco_mean",
                          "r2":"r2_loco_mean"})
    )

    summary = temp.merge(loco_mean, on="model", how="outer")
    summary.to_csv(TABLE_DIR / "table_taskA.csv", index=False)

    # LaTeX
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Task A — Outcome prediction baseline results. "
        r"RMSE and MAE in percentage points. "
        r"Temporal split: train 2015--2021, test 2022--2024. "
        r"LOCO: mean over 12 held-out-country folds.}",
        r"\label{tab:taskA}",
        r"\begin{tabular}{lrrrrrr}",
        r"\toprule",
        r"Model & \multicolumn{3}{c}{Temporal} & "
        r"\multicolumn{3}{c}{LOCO (mean)} \\",
        r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
        r"& RMSE & MAE & R$^2$ & RMSE & MAE & R$^2$ \\",
        r"\midrule",
    ]
    for _, row in summary.iterrows():
        def f(v): return f"{v:.3f}" if pd.notna(v) else "--"
        lines.append(
            f"{row['model']} & {f(row.get('rmse_temp'))} & "
            f"{f(row.get('mae_temp'))} & {f(row.get('r2_temp'))} & "
            f"{f(row.get('rmse_loco_mean'))} & {f(row.get('mae_loco_mean'))} & "
            f"{f(row.get('r2_loco_mean'))} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABLE_DIR / "table_taskA.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    print(f"  [OK] table_taskA.csv / .tex")
    print(f"\n[done] Results : {RESULT_DIR.resolve()}")
    print(f"[done] Tables  : {TABLE_DIR.resolve()}")
    print(f"[done] Figures : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/06_task_B_trajectory_classification.py")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, temporal, loco = load_data()

    panel_lag    = make_lagged_features(panel)
    feature_cols = get_feature_cols(panel_lag)
    models       = get_models()

    print("\n[step 2] Models:", list(models.keys()))

    # Evaluations
    temporal_records = run_temporal(panel_lag, temporal, feature_cols, models)
    loco_records     = run_loco(panel_lag, loco, feature_cols, models)
    all_records      = temporal_records + loco_records

    # Feature importance
    print("\n[step 4] Feature importance (Random Forest) …")
    imp = get_feature_importance(panel_lag, temporal, feature_cols)
    print("  Top 5 features:")
    for _, row in imp.head(5).iterrows():
        print(f"    {row['label']:<30} {row['importance']:.4f}")

    # Figures
    print("\n[step 5] Figures …")
    plot_rmse_comparison(all_records)
    plot_actual_vs_pred(panel_lag, temporal, feature_cols)
    plot_feature_importance(imp)

    # Save
    print("\n[step 6] Saving results …")
    save_results(all_records)


if __name__ == "__main__":
    main()